# 03. DPO: SFT 済みアダプタを HF Hub から読んで preference 学習

- 入力: 02 で作った LoRA アダプタ (HF Hub の private repo)
- データ: 同じ大喜利データを `(prompt, chosen=高score回答, rejected=低score回答)` のペアに変換
- 学習: TRL `DPOTrainer` (Unsloth `PatchDPOTrainer` で高速化)
- 出力: DPO 済み LoRA アダプタを HF Hub に push

## Colab セットアップ

In [ ]:
import sys, os
IS_COLAB = "google.colab" in sys.modules
print("Colab:", IS_COLAB)

if IS_COLAB:
    REPO_URL = "https://github.com/watamoo/oogiri-AI.git"
    REPO_DIR = "/content/oogiri-AI"
    if os.path.exists(REPO_DIR):
        get_ipython().system(f"cd {REPO_DIR} && git pull --ff-only")
    else:
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", f"{REPO_DIR}/notebooks")

    get_ipython().run_line_magic("pip", "install -q --upgrade pip")
    get_ipython().run_line_magic("pip", 'install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
    get_ipython().run_line_magic("pip", 'install -q --no-deps "trl<0.20.0" peft accelerate bitsandbytes')
    get_ipython().run_line_magic("pip", "install -q datasets pandas pyarrow japanize-matplotlib")

In [ ]:
from pathlib import Path
import sys
import warnings

if "google.colab" not in sys.modules:
    try:
        get_ipython().run_line_magic("load_ext", "autoreload")
        get_ipython().run_line_magic("autoreload", "2")
    except Exception as e:
        print("autoreload skipped:", e)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
try:
    import transformers
    transformers.logging.set_verbosity_error()
except ImportError:
    pass

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CHECKPOINT_DIR = ROOT / "checkpoints" / "dpo_qwen3_8b"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ROOT

## HF Hub 認証

private repo の SFT アダプタを読むのに必要。Colabセッション切れたら毎回実行。

In [ ]:
from huggingface_hub import login

login()

## ベースアダプタ選択 + モデルロード

`BASE_ADAPTER_REPO` を切り替えるだけで別の SFT 版から DPO を始められる。

In [ ]:
# DPO の出発点となるアダプタ (HF Hub repo 名)
BASE_ADAPTER_REPO = "watamoo/oogiri-sft-qwen3-8b-lora"
# 他候補がある場合はここを差し替え
# BASE_ADAPTER_REPO = "watamoo/oogiri-sft-qwen3-8b-lora-v2"

MAX_SEQ_LEN = 1024  # DPO は chosen+rejected 両方を1stepで処理するので SFT より少し長めに取る

from unsloth import FastLanguageModel, PatchDPOTrainer
import torch

# DPOTrainer に Unsloth 最適化を当てるためのパッチ (DPOTrainer import 前に呼ぶ)
PatchDPOTrainer()

# adapter repo を直接指定すると Unsloth が base + adapter を両方ロードする
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_ADAPTER_REPO,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_training(model)

## DPO データ準備

同じお題内で score 上位 → chosen、下位 → rejected として組み合わせる。`min_score_gap` で score 差が小さいノイズペアを除外。

In [ ]:
from src.data import load_oogiri_dpo

# top1 vs last1〜last5 の構成 (1お題あたり最大5ペア)
train_ds, eval_ds = load_oogiri_dpo(
    eval_ratio=0.1,
    seed=42,
    chosen_top_k=1,
    rejected_bottom_k=5,
    min_score_gap=10,
)
print("train pairs:", len(train_ds), "| eval pairs:", len(eval_ds))
print("--- sample ---")
ex = train_ds[0]
for k, v in ex.items():
    print(k, ":", v)

## DPOTrainer 設定

- `beta=0.1` 標準値
- `learning_rate=1e-5` (SFTの 1/10)
- `num_train_epochs=3` 学習量増量
- ref_model は LoRA を一時的に切ったベースモデルが暗黙的に使われる

In [ ]:
from trl import DPOTrainer, DPOConfig

trainer = DPOTrainer(
    model=model,
    ref_model=None,  # LoRA を一時 off にしたベースモデルが ref として使われる
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=DPOConfig(
        max_length=MAX_SEQ_LEN,
        max_prompt_length=512,
        beta=0.1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=4,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=1e-5,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        seed=3407,
        output_dir=str(CHECKPOINT_DIR),
        report_to="none",
        # FA2 が壊れてる Colab 環境では padding-free を切る (02_sft と同様)
        padding_free=False,
    ),
)

In [ ]:
# 学習前のVRAM
gpu = torch.cuda.get_device_properties(0)
used = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total = round(gpu.total_memory / 1024**3, 2)
print("GPU:", gpu.name, "|", used, "/", total, "GB used")

## 学習

In [ ]:
trainer_stats = trainer.train()
trainer_stats

In [ ]:
peak = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print("Peak VRAM:", peak, "GB")

## 学習中のLoss推移

DPO loss と reward margin (chosen vs rejected) を見る。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
train_log = logs[logs["loss"].notna()] if "loss" in logs.columns else pd.DataFrame()
eval_log = logs[logs["eval_loss"].notna()] if "eval_loss" in logs.columns else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
if not train_log.empty:
    axes[0].plot(train_log["step"], train_log["loss"],
                 label="Train loss", marker="o", markersize=3, linewidth=1)
if not eval_log.empty:
    axes[0].plot(eval_log["step"], eval_log["eval_loss"],
                 label="Eval loss", marker="x", markersize=10, linestyle="--")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("DPO loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Reward margin (chosen - rejected)
margin_col = "rewards/margins"
if margin_col in logs.columns:
    margin_log = logs[logs[margin_col].notna()]
    axes[1].plot(margin_log["step"], margin_log[margin_col],
                 label="Reward margin (chosen - rejected)", marker="o", markersize=3, linewidth=1)
    axes[1].axhline(0, color="gray", linewidth=0.5)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Margin")
    axes[1].set_title("Reward margin")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 学習後の生成サンプル

validation お題で生成して、DPO 効果を目視確認する。

In [ ]:
from src.data import load_oogiri_t2t, SYSTEM_PROMPT

model.generation_config.max_length = None
FastLanguageModel.for_inference(model)

# 比較用に元の eval_df も持ってくる (validation のお題と高score回答)
_, eval_df = load_oogiri_t2t(eval_ratio=0.1, seed=42, top_k=5)
eval_odai_df = (
    eval_df.drop_duplicates(subset=["odai_id"])[["odai_id", "odai"]]
    .reset_index(drop=True)
)
print("validationお題数:", len(eval_odai_df))

N_SAMPLES_PER_ODAI = 3

for _, row in eval_odai_df.iterrows():
    odai = row["odai"]
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": odai},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    ).to("cuda")

    refs = (
        eval_df[eval_df["odai_id"] == row["odai_id"]]
        .sort_values("score", ascending=False)
        .head(3)[["text", "score"]]
    )

    print("=" * 60)
    print("お題:", odai)
    print("-- 参考(データ中の高score回答) --")
    for _, r in refs.iterrows():
        print(f"  [score={r['score']}] {r['text']}")
    print("-- DPOモデル生成 --")
    for i in range(N_SAMPLES_PER_ODAI):
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=64,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
        )
        gen = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        print(f"  [{i+1}] {gen.strip()}")
    print()

## DPO アダプタ保存 (HF Hub に push)

In [ ]:
DPO_REPO = "watamoo/oogiri-dpo-qwen3-8b-lora"

save_path = CHECKPOINT_DIR / "adapter"
model.save_pretrained(str(save_path))
tokenizer.save_pretrained(str(save_path))
print("local saved:", save_path)

model.push_to_hub(DPO_REPO, private=True)
tokenizer.push_to_hub(DPO_REPO, private=True)
print("pushed to:", f"https://huggingface.co/{DPO_REPO}")